# 02H — Amino-acid landscape


In [3]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📊 aa_pos histogram


In [5]:
tmp = d[['aa_pos_num']].dropna()
fig = px.histogram(tmp, x='aa_pos_num', nbins=70, title='aa_pos histogram')
fig.show()


## 2. 📊 aa_pos vs phenotype


In [6]:
tmp = d[['phenotype', 'aa_pos_num']].dropna()
fig = px.box(tmp, x='phenotype', y='aa_pos_num', points='outliers', title='aa_pos vs phenotype')
fig.show()


## 3. 🧪 KS aa_pos ⭐


In [7]:
vals_dmd = d.loc[d['phenotype'] == 'DMD', 'aa_pos_num'].dropna()
vals_bmd = d.loc[d['phenotype'] == 'BMD', 'aa_pos_num'].dropna()
stat, p = ks_2samp(vals_dmd, vals_bmd)
display(pd.Series({'n_dmd': len(vals_dmd), 'n_bmd': len(vals_bmd), 'ks_stat': stat, 'p_value': p}).to_frame('value'))


,value
n_dmd,2783.000000
n_bmd,286.000000
ks_stat,0.040700
p_value,0.766411


## 4. 📊 aa_pos vs REVEL


In [8]:
tmp = d[['aa_pos_num', 'revel_num']].dropna()
fig = px.scatter(tmp, x='aa_pos_num', y='revel_num', opacity=0.5, title='aa_pos vs REVEL')
fig.show()


## 5. 🧪 Spearman aa_pos vs REVEL


In [9]:
tmp, rho, p = spearman_xy(d, 'aa_pos_num', 'revel_num')
display(pd.Series({'n': len(tmp), 'spearman_rho': rho, 'p_value': p}).to_frame('value'))


,value
n,2.273000e+03
spearman_rho,-2.779939e-01
p_value,1.300387e-41


## 6. 📊 aa_pos vs mutation_type


In [10]:
top = d['mutation_type'].value_counts().head(6).index
tmp = d[d['mutation_type'].isin(top)][['mutation_type', 'aa_pos_num']].dropna()
fig = px.box(tmp, x='mutation_type', y='aa_pos_num', points='outliers', title='aa_pos vs mutation_type')
fig.show()


## 7. 🧪 Kruskal aa_pos ~ mutation


In [11]:
tmp = d[d['mutation_type'].isin(d['mutation_type'].value_counts()[lambda s: s >= 30].index)]
names, groups, stat, p = kruskal_group(tmp, 'mutation_type', 'aa_pos_num')
display(pd.Series({'n_groups': len(names), 'kruskal_h': stat, 'p_value': p}).to_frame('value'))


,value
n_groups,5.000000
kruskal_h,12.492463
p_value,0.014041


## 8. 📊 protein density plot


In [12]:
tmp = d[['aa_pos_num']].dropna()
fig = px.histogram(tmp, x='aa_pos_num', nbins=100, histnorm='probability density', title='Protein position density')
fig.show()


## 9. 📊 pathogenic density protein


In [13]:
tmp = d[['aa_pos_num', 'pathogenic']].dropna()
fig = px.histogram(tmp, x='aa_pos_num', color='pathogenic', nbins=100, histnorm='probability density', barmode='overlay', opacity=0.6, title='Protein density by pathogenic status')
fig.show()


## 10. 📊 domain overlay density 📖


In [14]:
top = d['domain_clean'].value_counts().head(6).index
tmp = d[d['domain_clean'].isin(top)][['aa_pos_num', 'domain_clean']].dropna()
fig = px.histogram(tmp, x='aa_pos_num', color='domain_clean', nbins=100, histnorm='probability density', barmode='overlay', opacity=0.5, title='Domain overlay density on protein')
fig.show()
